mix them pixels yooo

In [84]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.multioutput import RegressorChain
import linear_mixing as lmx

In [2]:
# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

In [211]:
# Plotting settings

plt.rcParams['font.family'] = 'serif' #sans-serif'

def calc_r2(true, predicted, note = ""):
    r2 = 1 - (np.sum((true - predicted)**2, axis = 0) / np.sum((true - np.mean(true, axis = 0))**2, axis = 0))
    print(f"R squared values for each component {note}:\n{r2}")
    return r2

def plot_multi_result(x_array, y_array, alpha, colour, cmap, fig_title, x_axis_label, y_axis_label, r2_values):
    mat_count = x_array.shape[1]
    fig, ax = plt.subplots(mat_count//2 + mat_count%2, 2, figsize = (8,8), sharex = True, sharey = True)
    fig.tight_layout(pad = 3)

    for col in range(mat_count):
        mat = x_array.columns[col].replace("_", " ")
        ax[col//2, col%2].scatter(x_array.iloc[:, col], y_array[:,col], alpha = alpha, c = colour, cmap = cmap)
        ax[col//2, col%2].set_xlabel(f"{x_axis_label} of {mat}")
        ax[col//2, col%2].set_ylabel (y_axis_label)
        ax[col//2, col%2].annotate(f"R2 = {r2_values.iloc[col]:.3f}", xy = (0.7, 0.1), xycoords = "axes fraction")
        ax[col//2, col%2].axline((0,0), slope = 1, color = "black", linestyle = "--")
        
    fig.suptitle(fig_title, y = 1)
    return fig, ax

# mixing pixels

In [ ]:
material_options = {
    # "kelp" : ['ecklonia', 'macrocystis', 'undaria'],
    # "brown_non_kelp" : ['acrocarpia', 'cystophora', 'carpophyllum', 
    #                 'durvillaea', 'hormosira', 'petalonia','phyllospora', 'sargassum', 'scytosiphon'],
    "brown_algae" : ['ecklonia', 'macrocystis', 'petalonia', 
                     'undaria','acrocarpia', 'cystophora', 'carpophyllum', 'durvillaea', 'hormosira', 'phyllospora', 'sargassum', 'scytosiphon'],
    "red_veg" : ['rhodophyte', 'filamentous_rhodophyte','frondose_rhodophyte',],
    "green_veg" : ['grass', 'ulva'],
    # "non_brown_autos" : ['grass', 'filamentous_rhodophyte',
    #                      'frondose_rhodophyte', 'rhodophyte', 'ulva'],
    # "all_non_kelp_autos" : ['acrocarpia', 'carpophyllum', 'cystophora', 
    #                         'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'petalonia', 'grass','hormosira','phyllospora', 'rhodophyte',  'sargassum','scytosiphon','ulva'],
    # "all_non_kelp" : ['acrocarpia', 'barnacle_shells', 'carpophyllum',
    #                   'cystophora', 'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'gravel', 'grass','hormosira','mussels','petalonia','phyllospora', 'rhodophyte', 'rock','sand', 'sargassum', 'scytosiphon', 'shell_litter','rock', 'ulva', 'worm_castings'],
    "mineral" : ['barnacle_shells', 'gravel', 'mussels', 'rock', 'sand', 
                  'shell_litter', 'worm_castings'],
    }
material_options.keys()

In [5]:
# #Assign fractional cover ranges with simulate within
# cover_ranges = {
#     "kelp" : (0,0.8),
#     #"brown_algae" : (0, 0.5), 
#     "brown_non_kelp" : (0,0.8),
#     "red_veg" : (0,0.2),
#     "green_veg" : (0, 0.5),
#     #"non_brown_autos" : (0, 40),
#     #"all_non_kelp_autos" : (0, 100),
#     #"all_non_kelp" : (0,100),
#     "mineral" : (0,0.7),}

cover_ranges = {
    # "kelp" : (0,1),
    # "brown_non_kelp" : (0,1),
    "brown_algae" : (0, 1),
    "red_veg" : (0,1),
    "green_veg" : (0, 1),
    #"non_brown_autos" : (0, 40),
    #"all_non_kelp_autos" : (0, 100),
    #"all_non_kelp" : (0,100),
    "mineral" : (0,1),}

In [6]:
# Load spectral data from file and prep
data = pd.read_csv("./data/processed/resampled/noisy_Sentinel_2_ABC_resampled.csv", index_col=0)
labels = pd.read_csv("data/labels_prepped.csv", index_col=0)
data = pd.concat([labels["Class"], data], axis = 1)

In [ ]:
# Load an prepare water pixel spectra from file
deep_waters = pd.read_csv("./S2_deep_water_agg.csv", index_col=0)
deep_waters["depth"] = "deep"
shallow_waters = pd.read_csv("./S2_shallow_agg.csv", index_col=0)
shallow_waters["depth"] = "shallow"
waters = pd.concat([deep_waters, shallow_waters], axis = 0, ignore_index= True).reset_index(drop = True)
water_spec = waters.iloc[:, 4:-4]
print(f"Total of {water_spec.shape[0]} water pixels loaded")
water_spec.drop_duplicates(inplace = True)
print(f"Total of {water_spec.shape[0]} unique water pixels")
water_spec= water_spec/10000


In [8]:
# Split spectra into training and testing sets
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size = 0.3, stratify=labels["Class"])

water_train, water_test = train_test_split(water_spec, test_size = 0.3)

In [167]:
# Initialize the pixel mixer
mixer = lmx.LinearMixing(materials = material_options, cover_ranges = cover_ranges, pixels = 10000, material_spectra= data_train, water_spectra = water_train)

In [168]:
# Mix pixels and store results
results, f = mixer.sim_many_pixels()

In [169]:
# Format results and clear old variables
simulated_pixels, pixel_fpcs = mixer.format_sim_results(results, f)
del results, f

In [ ]:
# Inspect simulated results
print(simulated_pixels.shape)
print(pixel_fpcs.head())

In [ ]:
# Plot the components of the simulated pixels

args = {'log' : True} 

mat_count = len(material_options.keys())
fig, ax = plt.subplots(mat_count//2 + mat_count%2, 2, figsize = (8,4))
fig.tight_layout(pad = 3)

for col in range(mat_count):
    ax[col//2, col%2].hist(pixel_fpcs.iloc[:,col], bins = 50, range = (0.00000000000, max(pixel_fpcs.iloc[:, col])), **args)
    ax[col//2, col%2].set_title(pixel_fpcs.columns[col].replace("_", " "))
    ax[col//2, col%2].set_xlabel("fractional cover")
    ax[col//2, col%2].set_ylabel("pixel count")

fig.suptitle("Distribution of simulated fractional covers", y = 1.02)
plt.show()

In [ ]:
plt.hist(pixel_fpcs['non-water_comps'])
plt.xlabel("number of non-water components")
plt.ylabel("Pixel count")
plt.suptitle("Count of non-water components")
# fig.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\simulated_pixel_component_counts.svg")

plt.show()


In [15]:
# Export simulated pixels

#simulated_pixels.to_csv("data/mixed_sims/kbrgm_simulated_pixels_18112025.csv")
#pixel_fpcs.to_csv("data/mixed_sims/kbrgm_pixel_fpcs_18112025.csv")

# Inspect  mixed pixels and spectra

In [ ]:
sim_data = pd.concat([simulated_pixels, pixel_fpcs], axis = 1)
sim_data.head()

In [224]:
maj_water = sim_data[sim_data["maj"]=="water"].iloc[:200,:]

In [ ]:
plt.plot(maj_water.iloc[:, :8], c = maj_water["water"], cmap = "viridis")

# Prepare data for classification

In [173]:
# Define what you simulated
label_level = "kbrgm"
keep_list = ["kelp", "water", "other_brown_alg", "mineral", "green_veg", "red_veg"]

# Add water to classifier training set 
sample_idx = np.random.choice(water_train.index, size = 1000, replace = False)
water_labels = pd.DataFrame("water", index = sample_idx, columns = labels.columns)
water_sample = water_train.loc[sample_idx, :]
water_sample.columns = data_train.columns[1:]
data_watered = pd.concat([data_train, water_sample], axis = 0)
labels_watered = pd.concat([labels_train, water_labels], axis = 0).set_index(data_watered.index)
data_watered = pd.concat([labels_watered, data_watered.iloc[:, 1:]], axis = 1)


In [174]:
labels_watered["brgm"] = labels_watered["kbrgm"].replace(to_replace=["kelp", "brown_non_kelp"], value = "browns")

In [175]:
# Filter training data to only mixed pixel components
data_watered = data_watered[data_watered[label_level].isin(keep_list)]
water_endmember = data_watered[label_level] == "water"


In [176]:
# Extract PCA components and transform dataset
deco = PCA(8)
deco.fit(simulated_pixels) #.iloc[:,8:]) #ENDMEMBERS?
# decomposed_endmembers = deco.transform(data_watered.iloc[:, 8:])
decomposed_mixpix = deco.transform(simulated_pixels)
comps = deco.components_

In [ ]:
plt.plot(comps, label = range(comps.shape[1]))
plt.legend()
plt.show()

In [178]:
pixel_fpcs["maj"] = pixel_fpcs.iloc[:, :-1].idxmax(axis = 1)
fpcs = pixel_fpcs.iloc[:, :-2]

In [179]:
# divide the dataset into training and testing
x_train, x_test, y_train, y_test = train_test_split(decomposed_mixpix, fpcs, train_size = 0.9, random_state = 4)
x_train = pd.DataFrame(x_train)
x_test = pd.DataFrame(x_test)

# Evaluate Model

In [ ]:
# K-Folds cross validation

kf = KFold(n_splits=5)
scores_per_target = []

for train_idx, val_idx in kf.split(x_train):
    X_train, X_val = x_train.iloc[train_idx, :], x_train.iloc[val_idx, :]
    Y_train, Y_val = y_train.iloc[train_idx, :], y_train.iloc[val_idx, :]
    
    chain = RegressorChain(RandomForestRegressor(min_samples_leaf = 30, max_features= 0.5, max_depth = 10, bootstrap = True)) #, max_depth = 10, max_features = 0.4))
    chain.fit(X_train, Y_train)
    Y_pred = chain.predict(X_val)
    
    # Calculate R² for each target
    fold_scores = [r2_score(Y_val.iloc[:, i], Y_pred[:, i]) 
                   for i in range(Y_val.shape[1])]
    scores_per_target.append(fold_scores)

# Average across folds
mean_scores = np.mean(scores_per_target, axis=0)
print(f"Mean R² per target: {mean_scores}")

In [ ]:
# Testing-Training error comparison
params = {
    "min_samples_leaf" : 30, 
    "max_features" : 0.5, 
    "max_depth" : 10, 
    "bootstrap" : True
}
rfreg_eval = RandomForestRegressor(**params)
chain_eval = RegressorChain(rfreg_eval, order = [4, 0, 3, 1, 2]).fit(x_train, y_train)
p_eval = chain_eval.predict(x_test)
g_eval = chain_eval.predict(x_train)


# Calc R squared on adjusted values
r2_p_eval = calc_r2(y_test, p_eval, "TESTING")
r2_g_eval = calc_r2(y_train, g_eval, "TRAINING")

# Perform Chained regression

In [189]:
chain = RegressorChain(RandomForestRegressor(**params), 
                       order = [4, 0, 3, 1, 2],
                       ).fit(x_train, y_train)
p = chain.predict(x_test)

In [ ]:
# Calc R squared
r2 = calc_r2(y_test, p)

In [ ]:
# Plot regressor results

plotting_params = {
    "x_array": y_test, 
    "y_array": p, 
    "alpha": p[:,-1],
    "colour": p[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Adjusted random forest regressor results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "r2_values" : r2
}
fig, ax = plot_multi_result(**plotting_params)  

In [ ]:
# Histogram of predicted total covers
p_sum = np.sum(p, axis = 1)
plt.hist(p_sum, bins = 50)
plt.xlabel("Total fractional area predicted per pixel")
plt.ylabel("Pixel count")
plt.show()

In [ ]:
# Adjust predicted area to 1

p_adj = p/ p_sum[:, None]

# Calc R squared on adjusted values
r2_adj = calc_r2(y_test, p_adj, "ADJUSTED TESTING")

In [ ]:
# Plot adjusted regressor results
plotting_params = {
    "x_array": y_test, 
    "y_array": p_adj, 
    "alpha": p_adj[:,-1],
    "colour": p_adj[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Adjusted random forest regressor results",
    "x_axis_label": "Fractional cover",
    "y_axis_label" : "Predicted cover", 
    "r2_values" : r2_adj
}
fig, ax = plot_multi_result(**plotting_params)

In [ ]:
#Calc regression reuslts on training
g = chain.predict(x_train)
r2_train = calc_r2(y_train, g, "TRAINING")

In [ ]:
# Plot regression results on Training

plotting_params = {
    "x_array": y_train, 
    "y_array": g, 
    "alpha": g[:,-1],
    "colour": g[:,-1],
    "cmap" : "viridis", 
    "fig_title" : "Regression results on TRAINING set",
    "x_axis_label": "fractional cover",
    "y_axis_label" : "Predicted cover", 
    "r2_values" : r2_train
}
fig, ax = plot_multi_result(**plotting_params)